### Imports

In [ ]:
import pandas as pd
import numpy as np
import ta
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import time
from datetime import datetime, timedelta

## Constants

In [ ]:
candle_stick_timeframe = 60 # as seconds
interval_min = int(candle_stick_timeframe/60)
prediction_candles = 10
duration = candle_stick_timeframe * prediction_candles
prediction_candles = -prediction_candles
accuracy_threshold = 0.6
candle_size_multiplier = 0
amount = 100
tick_symbol = "R_10"
count = 5000
data_dict ={
  "ticks_history":tick_symbol,
  "adjust_start_time": 1,
  "count": count,
  "end": "latest",
  "granularity":candle_stick_timeframe,
  "style": "candles"
}
duplicate_num = int(-prediction_candles -1)
# the number of times each column is to be repeated in the back story... 
# the total number of the same columns will hence be duplicate_num + 1 . In this case 6.

### Initialisation

### Download the data
Download the data once and train the model, the next data is only downloaded as 2 rows only and appended to already existing dataset for feature extraction and transformation

In [ ]:
df = pd.read_csv("ETH_USD_1HR.csv")
del df['epoch']
df

In [ ]:
data_history = df.copy()
data_history

## Feature Engineering
Here we introduce all the features we need (indicators, std, price action etc..)
* roc
* willims r indicator
* ATr
* cci
* stddev
* Aroon
* ema_9
* ema_20
* ema_50,
* ADX,
* RSI
* Candle size

### Trend Indicators
* EMA_9, EMA_20, EMA_50, Aroon indicator, adx indicator, cci indicator,

In [ ]:
ema_9 = ta.trend.EMAIndicator(close=df['close'], window=9)
ema_20 = ta.trend.EMAIndicator(close=df['close'], window=20)
ema_50 = ta.trend.EMAIndicator(close=df['close'], window=50)
df = df.assign(ema_9 = ema_9.ema_indicator())
df = df.assign(ema_20 = ema_20.ema_indicator())
df = df.assign(ema_50 = ema_50.ema_indicator())
aroon = ta.trend.AroonIndicator(high=df['high'], low=df['low'], window=14).aroon_indicator()
df = df.assign(aroon=aroon)
adx = ta.trend.ADXIndicator(high=df['high'],low=df['low'], close=df['close'], window=14).adx()
df = df.assign(adx=adx)
cci = ta.trend.CCIIndicator(high=df['high'], low=df['low'], close=df['close'], window=14).cci()
df = df.assign(cci=cci)

### Momentum Indicators
* Rsi, candle size, ROC,Williams %r

In [ ]:
candle_size = abs(df['close'] - df['open'])
df = df.assign(candle_size=candle_size)
rsi_value = ta.momentum.RSIIndicator(close=df['close'], window=14)
df = df.assign(rsi = rsi_value.rsi())
roc = ta.momentum.ROCIndicator(close=df['close'], window=5).roc()
df = df.assign(roc=roc)
williams = ta.momentum.WilliamsRIndicator(high=df['high'], low=df['low'], close=df['close'], lbp=14).williams_r()
df = df.assign(williams = williams)

## Volatility Indicators
* ATR indicator, BOllinger Bands Parameters, 

In [ ]:
atr = ta.volatility.AverageTrueRange(high=df['high'],
                                     low=df['low'], 
                                     close=df['close'],
                                     window=14).average_true_range()
df = df.assign(atr=atr)
bol_width = ta.volatility.BollingerBands(close=df['close'], window=20, window_dev=2).bollinger_wband()
df = df.assign(bol_width=bol_width)
bol_above = ta.volatility.BollingerBands(close=df['close'], window=20, window_dev=2).bollinger_hband_indicator()
df = df.assign(bol_above=bol_above)
bol_below = ta.volatility.BollingerBands(close=df['close'], window=20, window_dev=2).bollinger_lband_indicator()
df = df.assign(bol_below=bol_below)
average_candle_size = df['candle_size'].rolling(window=14).mean()
df = df.assign(average_candle_size = average_candle_size)
df

In [ ]:
dif = df.copy()
dif

In [ ]:
for column in dif.columns: # shift 1,2,3,4
    num = list(range(1,duplicate_num+1))
    new_columns = {}
    for n in num:
        new_column_name = f"{column}_{n}"
        # Assign the shifted values to the new column name in the dictionary
        new_columns[new_column_name] = df[column].shift(n)
        df = df.assign(**new_columns)

In [ ]:
df.tail()

## Calculate the target class


In [ ]:
prediction_candles

In [ ]:
predicted_price = df['close'].shift(prediction_candles)
df = df.assign(predicted_price = predicted_price)
df = df.dropna()
target = np.where(
    df['predicted_price'] > (df['close']),1,0)
  
del df['predicted_price']
df = df.assign(target=target)
df

In [ ]:
df.target.unique()

In [ ]:
x = df.drop(['target'], axis =1)
y = df['target']
# ext_data = x.loc[[544]]
# # y_external = y.loc[[544]]
# # y_external
# ext_data

In [ ]:
random_state = 1
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=1)
model = RandomForestClassifier(n_estimators=1000, random_state=1) # keep it 100 for faster execution
model.fit(x_train, y_train)

In [ ]:
y_pred = model.predict(x_test)
acc = accuracy_score(y_test, y_pred)
acc

In [ ]:
# model.predict(ext_data)

In [ ]:
# model.predict_proba(ext_data)

In [ ]:
y_probab = model.predict_proba(x_test)
compare = pd.DataFrame(y_probab)
new_x = np.where(y_probab[:, 0] >= accuracy_threshold, 1, 0)
new_y = np.where(y_probab[:, 1] >= accuracy_threshold, 1, 0)
true_value = np.array(y_test)
compare = compare.assign(new_x=new_x)
compare = compare.assign(new_y=new_y)
compare = compare.assign(true_value=true_value)
fall_value = compare['new_x'].value_counts()[1]
rise_value = compare['new_y'].value_counts()[1]
tp_n = np.sum((y_test==1) &(new_y==1))
tp = tp_n/rise_value * 100
tn_n = np.sum((y_test==0) & (new_x ==1))
tn = tn_n/fall_value * 100

In [ ]:
compare

In [ ]:
rise_value, fall_value

In [ ]:
print("Number of all signals/All Candles Received: ",len(y_pred))
print(f"Number of all CALLE Trade Taken: {rise_value} Number of Correct Trades: {tp_n}")
print(f"Number of PUTE trades Taken: {fall_value} Number of correct trades: {tn_n}")
print("Total Number of Trades Takesn: ", tn_n + tp_n)
print("Percentage of candles traded: ",(tn_n+tp_n)/len(y_pred)*100)
print("Model Accuracy is: ",acc)
print("Rise Accuracy is: ",tp)
print("Fall Accuracy is: ", tn)

In [ ]:
strategy_accuracy = (tp*rise_value + tn * fall_value)/(rise_value+fall_value)
strategy_accuracy

### Deployment
* Stream just two rows of data, add the last row to first_copy data, then transform the data to fully form.
* Take the last row as the new data to predicted in a new variable

In [ ]:
344*66